# Лабораторная работа № 2. Основной сервер


## Функции сервера


In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import xmlrpc.client as client
import datetime
import csv
import functools
import numpy as np
import pickle


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


server = SimpleXMLRPCServer(("localhost", 8008), 
                            requestHandler=RequestHandler, logRequests=False, allow_none=True)

class TimeoutTransport(client.Transport):
    def make_connection(self, host):
        connection = super().make_connection(host)
        connection.timeout = 1
        return connection


STATS_URL = "http://localhost:8018"


def register_operation(function, name):
    @functools.wraps(function)
    def logged(*args):
        event_time = datetime.datetime.now().isoformat(timespec='microseconds')
        try:
            return function(*args)
        finally:
            try:
                with client.ServerProxy(STATS_URL, transport=TimeoutTransport()) as stats:
                    stats.add_log(name, event_time)
            except (OSError, client.Error):
                pass
    server.register_function(logged, name)


def blacklist_rows():
    with open('bad_boys2.csv', encoding='utf-8-sig', newline='') as file:
        return list(csv.DictReader(file))


def normalize(value):
    return value.strip().casefold()


def one_letter_difference(left, right):
    left, right = normalize(left), normalize(right)
    if left == right or abs(len(left) - len(right)) > 1:
        return False
    if len(left) == len(right):
        return sum(a != b for a, b in zip(left, right)) == 1
    shorter, longer = sorted((left, right), key=len)
    return any(longer[:i] + longer[i + 1:] == shorter for i in range(len(longer)))


def black_list_check_full(surname, name, patronymic, birth_date):
    entered = dict(zip(
        ('Surname', 'Name', 'Patronym', 'Birth'),
        (surname, name, patronymic, birth_date)
    ))

    exact = False
    similar = []

    for row in blacklist_rows():
        if row['Birth'].strip() != birth_date.strip():
            continue

        pairs = [(row[field], entered[field]) for field in ('Surname', 'Name', 'Patronym')]
        different = [(a, b) for a, b in pairs if normalize(a) != normalize(b)]

        if not different:
            exact = True
            continue

        if all(one_letter_difference(a, b) for a, b in different):
            similar.append(row)

    if exact:
        return {'data': entered, 'similar': similar, 'status': 'yes'}

    if similar:
        return {'data': similar, 'status': 'similar'}

    return {'data': entered, 'status': 'no'}

register_operation(black_list_check_full, 'black_list_check_full')


def binarize_image(bin_data, threshold):
    if isinstance(threshold, bool) or not isinstance(threshold, int):
        raise TypeError('Threshold must be integer')
    if not 1 <= threshold <= 255:
        raise ValueError('Threshold must be between 1 and 255')
    image = pickle.loads(bin_data.data)
    if image.size == 0:
        raise ValueError('Empty image data')
    if image.ndim == 2:
        brightness = image
    elif image.ndim == 3 and image.shape[2] == 3:
        brightness = image.astype(np.int64).sum(axis=2) // 3
    else:
        raise ValueError('Unsupported image format')
    binary = np.where(brightness > threshold, 255, 0).astype(np.uint8)
    if image.ndim == 3:
        binary = np.repeat(binary[:, :, None], 3, axis=2)
    return client.Binary(pickle.dumps(binary))

register_operation(binarize_image, 'binarize_image')


def estimate_cloudiness(bin_data, threshold):
    binary = binarize_image(bin_data, threshold)
    image = pickle.loads(binary.data)
    mask = image if image.ndim == 2 else image[:, :, 0]
    percent = float(np.count_nonzero(mask == 255) * 100 / mask.size)
    return {'image': binary, 'percent': percent}


register_operation(estimate_cloudiness, 'estimate_cloudiness')


def ping():
    return True

register_operation(ping, 'ping')


def now():
    return datetime.datetime.now()

register_operation(now, 'now')


def show_type(arg):
    return (str(arg), str(type(arg)), arg)

register_operation(show_type, 'type')


def test_sum(a, b):
    return a + b

register_operation(test_sum, 'sum')


def test_pow(a, b):
    return a ** b

register_operation(test_pow, 'pow')


def send_back_binary(bin_data):
    data = bin_data.data
    return client.Binary(data)

register_operation(send_back_binary, 'send_back_binary')


def send_back_inversion(bin_data):
    img_arr = pickle.loads(bin_data.data)

    if len(img_arr.shape) == 2:
        height, width = img_arr.shape
        for i in range(height):
            for j in range(width):
                img_arr[i][j] = 255 - img_arr[i][j]
    elif len(img_arr.shape) == 3:
        height, width, channels = img_arr.shape
        for i in range(height):
            for j in range(width):
                for k in range(channels):
                    img_arr[i][j][k] = 255 - img_arr[i][j][k]
    else:
        raise ValueError(f"Unsupported image format: {img_arr.shape}")

    pimg = pickle.dumps(img_arr)
    return client.Binary(pimg)

register_operation(send_back_inversion, 'color_inversion')


def flip_vertical(bin_data):
    img_arr = pickle.loads(bin_data.data)
    
    if len(img_arr.shape) == 2:
        height, width = img_arr.shape
        for i in range(height):
            for j in range(width // 2):
                temp = img_arr[i][j]
                img_arr[i][j] = img_arr[i][width - 1 - j]
                img_arr[i][width - 1 - j] = temp
                
    elif len(img_arr.shape) == 3:
        height, width, channels = img_arr.shape
        for i in range(height):
            for j in range(width // 2):
                for k in range(channels):
                    temp = img_arr[i][j][k]
                    img_arr[i][j][k] = img_arr[i][width - 1 - j][k]
                    img_arr[i][width - 1 - j][k] = temp
    else:
        raise ValueError("Unsupported image format")
    
    pimg = pickle.dumps(img_arr)
    return client.Binary(pimg)

register_operation(flip_vertical, 'flip_vertical')


def black_list_check(sname):
    similar = []
    for row in blacklist_rows():
        surname = row['Surname']
        if normalize(surname) == normalize(sname):
            return {'data': sname, 'status': 'yes'}
        if one_letter_difference(surname, sname) and surname not in similar:
            similar.append(surname)
    return {'data': similar if similar else sname,
            'status': 'similar' if similar else 'no'}


register_operation(black_list_check, 'black_list_check')

server.serve_forever()